# 📑 AML Payment Raw Data
 
## 🔗 Links
- **[BRD AML Payment Raw Data](https://cepalhellas.atlassian.net/wiki/spaces/DE/pages/897318915/BRD+AML+Raw+Payment+Data+Cloud+Implementation)**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Ticket related to the report](https://cepalhellas.atlassian.net/browse/DBTC-1604)**  
  Ticket regarding the addition of the 'Payer' field to the report.
---



## Imports

In [0]:
##aws_utils
import boto3
from botocore.exceptions import NoCredentialsError

from pyspark.dbutils import DBUtils
from pyspark.sql import SparkSession
 
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:  
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try: 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']

# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  

    env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env_prefix


In [0]:
env = get_env_prefix()
#env= 'p' ## Read from production
#print(env)

### Businessdate

In [0]:
Businessdate = spark.sql(f"""                    
                        select max(BusinessDate) as BusinessDate
                        from `{env}-cdh`.derivative.dcollection           
                    """).collect()[0]["BusinessDate"]
#Businessdate='2026-07-17'
Businessdate = str(Businessdate)
#print(Businessdate)

### TempView to Unify Customers

In [0]:
# 1) Filter each table by BusinessDate FIRST (temp views)
# Call get_all_exposures ONCE via SELECT * to avoid COLUMN_ORDINAL_OUT_OF_BOUNDS
# and improve performance (1 function call instead of 2)
spark.sql(f"""
SELECT *
FROM `{env}-cdh`.primary.get_all_exposures(DATE({Businessdate!r}))
WHERE SourceSystem='profits'
""").createOrReplaceTempView("_v_exposure_raw")

spark.sql(f"""
SELECT
    CDI, ExposureKey, PortfolioName,
    PartyRelationshipType, PartyName, PartyTaxRegistrationNumber,
    PrimaryPartyFlag, {Businessdate!r} AS RunDate
FROM _v_exposure_raw
""").createOrReplaceTempView("v_exposure_bd")

spark.sql(f"""
SELECT
    CDI, ExposureKey,
    PartyRelationshipType, PartyName, {Businessdate!r} AS RunDate
FROM _v_exposure_raw
WHERE PrimaryPartyFlag = true
""").createOrReplaceTempView("v_exposure_bd_main")

spark.sql(f"""

SELECT CDI, TaxRegistrationNumber, PartyType
FROM `{env}-cdh`.primary.party
WHERE BusinessDate = '{Businessdate}' and SourceSystem='profits'
""").createOrReplaceTempView("v_party_bd")

spark.sql(f"""

SELECT PortfolioName, Prefix
FROM `{env}-cdh`.primary.portfolio
WHERE BusinessDate = '{Businessdate}'
""").createOrReplaceTempView("v_portfolio_bd")


# 2) Now join the filtered views instead of the base tables

aml_temp_df = spark.sql(f"""
WITH t AS (
    SELECT
        exp.CDI        AS UnifiedCustomerId,
        por.Prefix     AS portfolio
    FROM v_exposure_bd exp
    JOIN v_party_bd p
        ON p.CDI = exp.CDI 
    LEFT JOIN v_portfolio_bd por
        ON por.portfolioName = exp.PortfolioName
),
t1 AS (
    SELECT
        UnifiedCustomerId,
        concat_ws(',', sort_array(collect_set(portfolio))) AS results
    FROM t
    GROUP BY UnifiedCustomerId
)
SELECT *
FROM t1

""")

aml_temp_df.createOrReplaceTempView("aml_temp_portfolios")


### AML Raw Data Query

In [0]:
# Reuse existing filtered views created earlier in the notebook:
# - v_exposure_bd
# - v_party_bd
# - v_portfolio_bd
# - aml_temp_portfolios
#
# We pre-filter tables by BusinessDate and reuse temp views
# to improve readability, reusability, and avoid repeating the same
# BusinessDate filter in every JOIN condition.

# Create filtered view for collection
# (BusinessDate was previously filtered in the WHERE clause)
spark.sql(f"""

SELECT
    PortfolioName, recorddate, actualdate,
    PayerCDI, ExposureKey, amount,
    dcollectionkey, paymenttype, PrimaryOwnerCDI
FROM `{env}-cdh`.derivative.dcollection
WHERE BusinessDate = '{Businessdate}'
AND CAST(recorddate AS date) >= '2019-01-01'
""").createOrReplaceTempView("v_collection_bd")

# Single-pass aggregation: CDIs + AFMs + DebtorDetails per ExposureKey (1 shuffle)
spark.sql("""
SELECT 
    ExposureKey,
    sort_array(collect_set(CDI))                        AS AllDebtorCDIs,
    sort_array(collect_set(PartyTaxRegistrationNumber))  AS AllDebtorAFMs,
    sort_array(
        collect_set(
            named_struct(
                'CDI',           CDI,
                'AFM',           PartyTaxRegistrationNumber,
                'PartyName',     PartyName,
                'RelationType',  PartyRelationshipType
            )
        )
    ) AS DebtorDetails
FROM v_exposure_bd
WHERE ExposureKey IN (SELECT DISTINCT ExposureKey FROM v_collection_bd)
GROUP BY ExposureKey
""").createOrReplaceTempView("v_debtor_info")


aml_df = spark.sql(f"""
SELECT DISTINCT
    getdate() as CepalDate
    ,c.PortfolioName
    ,CAST(c.recorddate AS date) AS RecordDate
    ,CAST(c.actualdate AS date) AS ActualDate
    ,c.PayerCDI AS PayerCDI              
    ,c.ExposureKey                 AS AccountNumber -- search for isnull(asset.AssetKey, p.LoanBankCode) to be sure in exposure notebooks

    ,c.amount                  AS PaymentAmountEuro
    -- ,exp.PartyTaxRegistrationNumber -- check if correct and if we need case null = cdi like on prem
    
    ,CASE
        WHEN COALESCE(p.TaxRegistrationNumber, CAST(p.CDI AS STRING)) LIKE '%000000000%'
            THEN CAST(p.CDI AS STRING)
        WHEN COALESCE(p.TaxRegistrationNumber, CAST(p.CDI AS STRING)) = ''
            THEN CAST(p.CDI AS STRING)
        ELSE COALESCE(p.TaxRegistrationNumber, CAST(p.CDI AS STRING))
    END AS TaxNumber

    
    ,c.dcollectionkey           AS TUN
    
    ,CASE WHEN c.PayerCDI IS NOT NULL THEN coalesce(exp.partyrelationshiptype,exp_main.partyrelationshiptype) END AS PayerRelationshipType
    ,CASE WHEN c.PayerCDI IS NOT NULL THEN coalesce(exp.partyname,exp_main.partyname) END AS PayerPartyName
    ,CASE WHEN c.PayerCDI IS NOT NULL THEN p.taxregistrationnumber END AS PayerTaxRegistrationNumber
    
    ,t1.results                AS CustomerPortfolios
    
    ,CASE
        WHEN p.partytype = 'Individual' THEN 'Person'
        ELSE 'Legal Form'
    END                       AS customer_type
    
    ,c.paymenttype      AS PaymentType
    -- Some values are Null in cloud, N/A on premise, should I add a CASE WHEN or leave it as is?
    ,c.PrimaryOwnerCDI  AS PrimaryOwnerCDI
    ,p_owner.taxregistrationnumber  AS PrimaryOwnerTaxRegistrationNumber
    ,di.AllDebtorCDIs
    ,di.AllDebtorAFMs
    ,di.DebtorDetails
FROM v_collection_bd c
LEFT JOIN v_exposure_bd exp
    ON exp.exposurekey = c.ExposureKey
    AND exp.cdi=c.PayerCDI
LEFT JOIN v_exposure_bd_main exp_main
    ON exp_main.exposurekey = c.ExposureKey
LEFT JOIN v_party_bd p
    ON p.cdi = coalesce(c.PayerCDI,exp_main.cdi,c.PrimaryOwnerCDI)
LEFT JOIN v_party_bd p_owner
    ON p_owner.cdi = c.PrimaryOwnerCDI    
LEFT JOIN v_debtor_info di
    ON di.ExposureKey = c.ExposureKey
LEFT JOIN aml_temp_portfolios t1
    ON t1.unifiedcustomerid = coalesce(c.PayerCDI,exp_main.cdi,c.PrimaryOwnerCDI)
--- AssetXParty not needed
""")


### Output

In [0]:
target_area = f"`{env}-cdh-delivery`.`aml`" 

target_table_path_amlpaymentrawdata = f"{target_area}.AMLPaymentRawData"


aml_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table_path_amlpaymentrawdata) # in order to write in delta_lake


spark.sql(f"REPAIR TABLE {target_table_path_amlpaymentrawdata} SYNC METADATA")